## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, EarlyStopping, get_measures

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Binary Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [7]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[ 33 103]


In [8]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[15 44]


In [9]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [10]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [11]:
train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8, shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

## Model & Training

In [12]:
model = h_ANFIS(
    input_size = 22,
    num_mfs = 15,
    outputs = 1,
    rule_reduced = True,
    output_type='binary',
    dtype=torch.float64
)

#model.init_premises(x_train)

In [13]:
loss_fn = nn.functional.binary_cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.01, 'weight_decay': 0.1}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [14]:
trainer = Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [15]:
trainer(model, train_loader, verbose=True)

Epoch:   1/500 - loss: 0.522381 - validation loss: 0.532952
Epoch:   2/500 - loss: 1.726073 - validation loss: 1.309890
Epoch:   3/500 - loss: 0.537902 - validation loss: 0.545846
Epoch:   4/500 - loss: 0.416406 - validation loss: 0.406952
Epoch:   5/500 - loss: 0.497657 - validation loss: 0.573911
Epoch:   6/500 - loss: 0.532719 - validation loss: 0.540647
Epoch:   7/500 - loss: 0.500809 - validation loss: 0.509205
Epoch:   8/500 - loss: 0.531763 - validation loss: 0.540481
Epoch:   9/500 - loss: 0.531813 - validation loss: 0.539857
Epoch:  10/500 - loss: 0.463095 - validation loss: 0.509167
Epoch:  11/500 - loss: 0.535711 - validation loss: 0.541355
Epoch:  12/500 - loss: 0.534723 - validation loss: 0.537633
Epoch:  13/500 - loss: 0.676882 - validation loss: 0.517993
Epoch:  14/500 - loss: 0.502267 - validation loss: 0.552449
Epoch:  15/500 - loss: 0.535308 - validation loss: 0.542286
Epoch:  16/500 - loss: 0.538539 - validation loss: 0.550436
Epoch:  17/500 - loss: 0.539548 - valida

In [16]:
train_measures = get_measures(model, x_train, y_train)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8308823529411765
Precision: 0.9
Recall: 0.8737864077669902
F1: 0.8866995073891626
Confusion Matrix: [[23 10]
 [13 90]]


In [17]:
test_measures = get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.8305084745762712
Precision: 0.8541666666666666
Recall: 0.9318181818181818
F1: 0.8913043478260869
Confusion Matrix: [[ 8  7]
 [ 3 41]]
